# Writing classes

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

We write our own complex number, `Complex`, and along the way we meet everything a class
needs.

**Objectives**

* write a class: `class`, `__init__`, `self`, methods;
* keep the object valid: **invariants** checked in the constructor;
* give it operators and a readable output with **special methods**;
* **test** a class against a reference.

In [ ]:
import math
import cmath

## 1. Design first

Before coding a class, decide its **interface** (what the user can do with it) and its
**state** (what it stores).

For `Complex` the interface is that of the built-in `complex`: real and imaginary parts,
modulus, conjugate, `+`, `*`, `/`, `==`, `print`.

The state is our choice. To make the point, we do **not** store the real and imaginary
parts, but the **modulus** and the **phase**: $z = r\,e^{i\phi}$.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Polar or Cartesian?</b></p>

Storing $(r, \phi)$ instead of $(x, y)$: which operations become easier, and which
harder? Would the **user** of the class notice the difference?

</div>

<details>
<summary><b>Solution</b></summary>

Products, quotients and powers are trivial in polar form ($r_1 r_2$, $\phi_1 + \phi_2$);
sums and differences need Cartesian form. The user should **not** notice: if the
interface is the same, the internal representation can change without breaking any code
that uses the class. This is **information hiding**, or encapsulation.

</details>

## 2. The class, the constructor and self

The minimal class:

In [ ]:
class Complex:
    """Complex number stored as modulus and phase."""

    def __init__(self, mod=0., phase=0.):
        self.mod = float(mod)
        self.phase = float(phase)

z = Complex(2., math.pi / 2)
print(z.mod, z.phase, type(z))

* `class Complex:` opens the definition; the methods are functions indented inside it.
* `__init__` is the **constructor**: Python calls it when we write `Complex(2., ...)`,
  after creating an empty object.
* `self` is **the object being built** (or, in any other method, the object on which the
  method is called). `self.mod = ...` creates the attribute `mod` of *that* object.
* We do not pass `self`: `Complex(2., 1.)` is `Complex.__init__(new_object, 2., 1.)`.

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Invalid objects</b></p>

What do these lines create? Are they valid complex numbers?

```python
a = Complex(-1., 0.)
b = Complex(1., 0.)
c = Complex(1., 2 * math.pi)
```

</div>

<details>
<summary><b>Solution</b></summary>

`a` has a negative modulus: it is not a valid polar representation. `b` and `c` are the
**same** number, $1$, with two different phases: `==` would say they differ.

The constructor is the place to enforce the **invariants**, the conditions that every
object must satisfy: $r \ge 0$, and $\phi$ reduced to $(-\pi, \pi]$. If they are
enforced there, no object can be **created** invalid.

</details>

## 3. The full class

Here is the class with its invariants, methods and operators. We go through it below.

In [ ]:
class Complex:
    """Complex number stored as modulus and phase."""

    def __init__(self, mod=0., phase=0.):
        if mod < 0:
            raise ValueError(f'the modulus must be non-negative, got {mod}')
        self.mod = float(mod)
        phase = math.remainder(float(phase), 2 * math.pi)     # in [-pi, pi]
        self.phase = math.pi if phase == -math.pi else phase  # in (-pi, pi]

    @classmethod
    def from_cartesian(cls, x, y):
        """Alternative constructor, from the real and imaginary parts."""
        return cls(math.hypot(x, y), math.atan2(y, x))

    def real(self):
        return self.mod * math.cos(self.phase)

    def imag(self):
        return self.mod * math.sin(self.phase)

    def conjugate(self):
        return Complex(self.mod, -self.phase)

    def __abs__(self):
        return self.mod

    def __mul__(self, other):
        return Complex(self.mod * other.mod, self.phase + other.phase)

    def __truediv__(self, other):
        if other.mod == 0:
            raise ZeroDivisionError('division by a zero Complex')
        return Complex(self.mod / other.mod, self.phase - other.phase)

    def __add__(self, other):
        return Complex.from_cartesian(self.real() + other.real(),
                                      self.imag() + other.imag())

    def __neg__(self):
        return Complex(self.mod, self.phase + math.pi)

    def __sub__(self, other):
        return self + (-other)

    def __repr__(self):
        return f'Complex({self.mod:.6g}, {self.phase:.6g})'

In [ ]:
x = Complex(1., 0.)
y = Complex(1., math.pi / 2)
print(x + y, x * y, abs(x + y))
print(Complex.from_cartesian(1., 1.))

In [ ]:
print(Complex(1., 2 * math.pi))     # the phase is reduced to 0
Complex(-1., 0.)                     # the invariant is enforced

### 3.1 Methods

A method is a function whose first argument is the object, `self`. Inside, the object's
attributes are reached as `self.mod`, `self.phase`, and other methods as `self.real()`.

`conjugate`, `__mul__` and `__add__` return a **new** `Complex`: they do not change
`self`. Like numbers, our complex numbers are **immutable**, by convention: no method
changes `self`. (Nothing stops a user from writing `z.mod = -1`; Python trusts you.)

### 3.2 Special methods

`x + y` calls `x.__add__(y)`; `abs(x)` calls `x.__abs__()`; `-x` calls `x.__neg__()`.
`__sub__` is written with `+` and `-`: reuse what already exists.

`print` calls `str`, which falls back on `__repr__`; so does the notebook display.

### 3.3 A second constructor: classmethod

`@classmethod` makes a method that receives the **class**, `cls`, instead of an object.
It is the standard way to offer alternative constructors, such as
`Complex.from_cartesian(x, y)` here, `dict.fromkeys` or `datetime.fromtimestamp`.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Mixing with numbers</b></p>

**Decide**: which of these work?

```python
z = Complex(2., 0.)
z * Complex(3., 0.)
z * 3
3 * z
```

</div>

In [ ]:
z = Complex(2., 0.)
print(z * Complex(3., 0.))

In [ ]:
print(z * 3)

In [ ]:
print(3 * z)

<details>
<summary><b>Solution</b></summary>

Only the first. `z * 3` calls `z.__mul__(3)`, which asks for `3.mod`:
`AttributeError`. `3 * z` calls `int.__mul__(3, z)`, which returns `NotImplemented`
because an `int` does not know our class; Python then tries `z.__rmul__(3)`, which does
not exist: `TypeError`.

The fix: convert numbers in `__mul__`, return `NotImplemented` (do not raise) for types
you do not know, so that the other operand can try, and add `__rmul__` for the
right-hand side:

```python
import numbers

def __mul__(self, other):
    if isinstance(other, numbers.Real):          # int, float, numpy numbers...
        other = Complex(abs(other), 0. if other >= 0 else math.pi)
    elif not isinstance(other, Complex):
        return NotImplemented                    # Python tries other.__rmul__
    return Complex(self.mod * other.mod, self.phase + other.phase)

__rmul__ = __mul__
```

`__add__` and `__truediv__` need the same recipe.

</details>

## 4. Equality

Without `__eq__`, `==` compares **identity**: two different objects are never equal,
even with the same values.

In [ ]:
print(Complex(1., 0.) == Complex(1., 0.))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Is it zero?</b></p>

We add the obvious `__eq__`, comparing modulus and phase. **Decide**: is
`Complex(1., 0.) - Complex(1., 0.) == Complex()` true?

```python
def __eq__(self, other):
    return self.mod == other.mod and self.phase == other.phase
```

</div>

In [ ]:
Complex.__eq__ = lambda self, other: self.mod == other.mod and self.phase == other.phase

print(Complex(1., 0.) - Complex(1., 0.))
print(Complex(1., 0.) - Complex(1., 0.) == Complex())

<details>
<summary><b>Solution</b></summary>

False. $1 - 1$ goes through $-1 = e^{i\pi}$, whose real part is `-1.0` but whose
imaginary part is `1.2e-16`: the result has a modulus of `1.2e-16`, not `0`. Exact
equality is fragile for computed floats, also inside a class.

(We attached `__eq__` to the existing class with an assignment, to keep the cell short;
in real code it goes inside the `class` block, and there it also makes the objects
unhashable unless you define `__hash__`. It should also `return NotImplemented` when
`other` is not a `Complex`, instead of failing on `other.mod`.)

</details>

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Should == use a tolerance?</b></p>

We could define `__eq__` with `math.isclose` on the real and imaginary parts. What do we
gain? What do we lose? Think of: is `a == b` and `b == c` still implying `a == c`?
What tolerance, absolute or relative? What does the built-in `complex` do?

</div>

<details>
<summary><b>Solution</b></summary>

**Gain**: `1 - 1 == 0` is true. **Loss**: equality is no longer **transitive** (`a` close to
`b`, `b` close to `c`, but `a` not close to `c`), and the tolerance is arbitrary: it
depends on the scale of the numbers. The built-in `complex` and `float` use exact
equality, and so does numpy: they leave the tolerance to the user, with
`math.isclose` / `np.isclose`. That is usually the better choice: keep `==` exact, and
provide (or use) an explicit `isclose`.

In polar form there is one more trap: a tolerance on `(mod, phase)` fails for zero (any
phase is valid) and near $\pm\pi$. Compare `abs(a - b)`, as `cmath.isclose` does, and
remember that a relative tolerance needs an `abs_tol` near zero.

</details>

## 5. Testing a class

The built-in `complex` is a perfect **reference**: every operation of `Complex` must agree
with it.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Test Complex</b></p>

Write a helper `to_complex(z)` that converts a `Complex` into a `complex`. Then write tests
that compare `+`, `-`, `*`, `/`, `abs` and `conjugate` with the built-in `complex`, for
many random numbers (or with hypothesis). Do they all pass? With which tolerance?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
import random

def to_complex(z):
    return complex(z.real(), z.imag())

def random_pair():
    a = complex(random.uniform(-5, 5), random.uniform(-5, 5))
    b = complex(random.uniform(-5, 5), random.uniform(-5, 5))
    za = Complex.from_cartesian(a.real, a.imag)
    zb = Complex.from_cartesian(b.real, b.imag)
    return a, b, za, zb

def close(z, c):
    return cmath.isclose(to_complex(z), c, rel_tol=1e-9, abs_tol=1e-12)

def test_operations():
    random.seed(0)                     # a failure must be reproducible
    for _ in range(1000):
        a, b, za, zb = random_pair()
        assert close(za + zb, a + b)
        assert close(za - zb, a - b)
        assert close(za * zb, a * b)
        assert close(za / zb, a / b)
        assert math.isclose(abs(za), abs(a))
        assert close(za.conjugate(), a.conjugate())

test_operations()
```

An `abs_tol` is needed: when the result is close to zero, a relative tolerance alone
fails.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[&] Collaborate — Vector3D, in pairs</b></p>

A 3D vector class, `Vector3D(x, y, z)`, with `+`, `-`, multiplication by a number (both
sides), `dot`, `cross`, `abs` (the norm), `unit` (the unit vector), `==` and a readable
`repr`.

* **A** writes the class.
* **B**, at the same time, writes the tests: properties ($\mathbf{a}\times\mathbf{b} =
  -\mathbf{b}\times\mathbf{a}$, $\mathbf{a}\cdot(\mathbf{a}\times\mathbf{b}) = 0$,
  $|\lambda\mathbf{a}| = |\lambda|\,|\mathbf{a}|$…) and a comparison with numpy.
* Cross: run B's tests on A's class. Then swap roles for a new method, `angle(other)`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
import numbers
import numpy as np

class Vector3D:
    """A vector in 3D space."""

    def __init__(self, x, y, z):
        self.x, self.y, self.z = float(x), float(y), float(z)

    def __add__(self, other):
        return Vector3D(self.x + other.x, self.y + other.y, self.z + other.z)

    def __neg__(self):
        return Vector3D(-self.x, -self.y, -self.z)

    def __sub__(self, other):
        return self + (-other)

    def __mul__(self, scalar):
        if not isinstance(scalar, numbers.Real):
            return NotImplemented
        return Vector3D(scalar * self.x, scalar * self.y, scalar * self.z)

    __rmul__ = __mul__

    def dot(self, other):
        return self.x * other.x + self.y * other.y + self.z * other.z

    def cross(self, other):
        return Vector3D(self.y * other.z - self.z * other.y,
                        self.z * other.x - self.x * other.z,
                        self.x * other.y - self.y * other.x)

    def __abs__(self):
        return math.sqrt(self.dot(self))

    def unit(self):
        return (1. / abs(self)) * self

    def __eq__(self, other):
        return (self.x, self.y, self.z) == (other.x, other.y, other.z)

    def __repr__(self):
        return f'Vector3D({self.x}, {self.y}, {self.z})'

def test_cross():
    a, b = Vector3D(1, 2, 3), Vector3D(-2, 0.5, 4)
    assert a.cross(b) == -(b.cross(a))   # exact: x - y == -(y - x) in IEEE
    assert math.isclose(a.dot(a.cross(b)), 0., abs_tol=1e-12)
    c = a.cross(b)
    assert np.allclose([c.x, c.y, c.z], np.cross([1, 2, 3], [-2, 0.5, 4]))
    assert math.isclose(abs(-3 * a), 3 * abs(a))
    assert math.isclose(abs(a.unit()), 1.)

test_cross()
```

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A histogram class</b></p>

Design and write a class `Histogram1D(nbins, xmin, xmax)` that:

* `fill(x)` adds one entry (and ignores, but counts, the values out of range);
* `counts` holds the number of entries per bin; `edges` the bin edges;
* `mean()` gives the mean of the entries, computed from the bins.

First write the interface and two tests, then the code. Check it against `np.histogram`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
import numpy as np

class Histogram1D:
    """Fixed-width 1D histogram."""

    def __init__(self, nbins, xmin, xmax):
        if nbins <= 0 or xmax <= xmin:
            raise ValueError('need nbins > 0 and xmax > xmin')
        self.nbins, self.xmin, self.xmax = nbins, xmin, xmax
        self.width = (xmax - xmin) / nbins
        self.edges = [xmin + i * self.width for i in range(nbins + 1)]
        self.counts = [0] * nbins
        self.outside = 0

    def fill(self, x):
        if not self.xmin <= x < self.xmax:
            self.outside += 1
            return
        i = int((x - self.xmin) / self.width)
        self.counts[min(i, self.nbins - 1)] += 1      # rounding just below xmax

    def mean(self):
        if sum(self.counts) == 0:
            raise ValueError('mean of an empty histogram')
        centers = [self.xmin + (i + 0.5) * self.width for i in range(self.nbins)]
        return sum(c * n for c, n in zip(centers, self.counts)) / sum(self.counts)

xs = np.random.default_rng(1).normal(0., 1., 10_000)
h = Histogram1D(20, -3., 3.)
for x in xs:
    h.fill(x)
ref, _ = np.histogram(xs, bins=20, range=(-3., 3.))
assert h.counts == list(ref)
assert h.outside == np.sum((xs < -3.) | (xs >= 3.))
```

Note the invariant in the constructor. `fill` is the only method that changes the state:
unlike `Complex`, a histogram is naturally **mutable**. And a convention: `np.histogram`
closes the last bin, $[\ldots, x_{max}]$; here $x_{max}$ is excluded.

</details>

***[+] Lookout.*** `@property` turns a method into something that is read like an attribute:
with `@property` on top of `def real(self)`, the user writes `z.real`, as with the
built-in `complex`, and the value is still computed from `mod` and `phase`. And
`@dataclass` (see [Why classes?](./fca_classes.ipynb)) writes `__init__`, `__repr__` and
`__eq__` for you. Python's data model, with all the special methods, is in the
[reference](https://docs.python.org/3/reference/datamodel.html#special-method-names).

## 6. Summary

* Design the interface first; the representation is an internal choice.
* Check the invariants in `__init__`.
* Special methods give operators and `print`; return `NotImplemented` for unknown types.
* Test against a reference.

<details>
<summary><b>[i] Cheat sheet — Writing classes</b></summary>

```python
class Name:
    """Docstring."""

    def __init__(self, a, b=0.):          # constructor
        if a < 0:                         # invariants, checked once
            raise ValueError('...')
        self.a, self.b = a, b             # attributes of this object

    def method(self, other):              # self = the object
        return Name(self.a + other.a)     # return a new object...

    def __add__(self, other): ...         # x + y
    def __eq__(self, other): ...          # x == y (default: identity)
    def __repr__(self): ...               # print(x), the notebook

    @classmethod
    def from_other(cls, data): ...        # alternative constructor
```

Design the interface first; check invariants in `__init__`; test against a reference.

</details>

### Check yourself

1. What is `self`, and why do we not pass it when calling a method?
2. Why check the invariants in `__init__`?
3. Without `__eq__`, what does `==` compare?
4. Why do we need `__rmul__` even when `z * 3` works?
5. What is a `classmethod` for?

<details>
<summary><b>Solutions</b></summary>

1. The object on which the method acts. `z.method(a)` is `Complex.method(z, a)`: Python
   passes it.
2. So that no object is born invalid: every other method can trust the state (as long as
   nobody assigns to the attributes directly).
3. Identity: whether both names refer to the same object.
4. `3 * z` calls `int.__mul__`, which returns `NotImplemented` because it does not know
   `Complex`; Python then looks for `z.__rmul__(3)`.
5. Alternative constructors: it receives the class (`cls`), not an object.

</details>